# LSTM v2 — Multi-horizon synthetic hourly-temperature experiment

This notebook trains one regularized LSTM on the Jena Climate dataset and predicts temperature at +1 hour, +2 hours, and +6 hours from a 72-hour history.

The threshold-contract output is a **synthetic Kalshi-style demonstration**. It is not an official Kalshi probability, does not use Kalshi market prices, and does not use an official settlement feed. The target is hourly mean temperature in degrees Celsius.


In [ ]:
from pathlib import Path
import json
import random
import urllib.request
import warnings
import zipfile

import joblib
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

OUTPUT_DIR = Path("/kaggle/working")
DATA_DIR = OUTPUT_DIR / "jena_data"
DATA_URL = "https://storage.googleapis.com/tensorflow/tf-keras-datasets/jena_climate_2009_2016.csv.zip"

TARGET = "T (degC)"
HISTORY_HOURS = 72
HORIZONS = np.array([1, 2, 6], dtype=int)
TRAIN_END = pd.Timestamp("2015-01-01")
TEST_START = pd.Timestamp("2016-01-01")
TEST_END = pd.Timestamp("2017-01-01")

BATCH_SIZE = 256
MAX_EPOCHS = 60
V1_MAE_6H = 1.229071021080017
V1_RMSE_6H = 1.614332692370595

print("TensorFlow:", tf.__version__)
print("Output directory:", OUTPUT_DIR)


In [ ]:
def find_attached_jena_csv():
    input_root = Path("/kaggle/input")
    if not input_root.exists():
        return None

    for candidate in sorted(input_root.rglob("*.csv")):
        try:
            header = pd.read_csv(candidate, nrows=2)
        except Exception:
            continue
        if {"Date Time", TARGET}.issubset(header.columns):
            return candidate
    return None


def load_jena_raw():
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    csv_path = DATA_DIR / "jena_climate_2009_2016.csv"

    if not csv_path.exists():
        zip_path = DATA_DIR / "jena_climate_2009_2016.csv.zip"
        try:
            print("Downloading the public Jena Climate dataset...")
            urllib.request.urlretrieve(DATA_URL, zip_path)
            with zipfile.ZipFile(zip_path) as archive:
                archive.extractall(DATA_DIR)
            extracted = list(DATA_DIR.glob("*.csv"))
            if extracted:
                csv_path = extracted[0]
        except Exception as error:
            fallback = find_attached_jena_csv()
            if fallback is None:
                raise RuntimeError(
                    "The Jena download failed. Enable Kaggle Internet and rerun, "
                    "or attach a CSV containing 'Date Time' and 'T (degC)' under "
                    "/kaggle/input, then rerun this cell."
                ) from error
            print("Download unavailable; using attached dataset:", fallback)
            csv_path = fallback

    frame = pd.read_csv(csv_path)
    if not {"Date Time", TARGET}.issubset(frame.columns):
        raise ValueError(
            f"Expected columns 'Date Time' and {TARGET!r}; found {list(frame.columns)}"
        )

    frame["Date Time"] = pd.to_datetime(
        frame["Date Time"],
        format="%d.%m.%Y %H:%M:%S",
        errors="coerce",
    )
    frame = (
        frame.dropna(subset=["Date Time"])
        .sort_values("Date Time")
        .drop_duplicates("Date Time", keep="last")
        .set_index("Date Time")
    )
    return frame


raw = load_jena_raw()
print("Raw shape:", raw.shape)
print("Raw range:", raw.index.min(), "to", raw.index.max())


In [ ]:
# Transform wind direction before averaging so the circular nature of degrees is respected.
wind_direction_columns = [
    column for column in raw.columns
    if column.strip().lower().startswith("wd")
]
if wind_direction_columns:
    wind_direction_column = wind_direction_columns[0]
    radians = np.deg2rad(raw[wind_direction_column])
    raw["wind_dir_sin"] = np.sin(radians)
    raw["wind_dir_cos"] = np.cos(radians)
else:
    wind_direction_column = None

numeric = raw.select_dtypes(include=[np.number])
hourly = numeric.resample("1h").mean()

if wind_direction_column and wind_direction_column in hourly.columns:
    hourly = hourly.drop(columns=[wind_direction_column])

hourly = hourly.replace([np.inf, -np.inf], np.nan)

hourly["hour_sin"] = np.sin(2 * np.pi * hourly.index.hour / 24.0)
hourly["hour_cos"] = np.cos(2 * np.pi * hourly.index.hour / 24.0)
hourly["day_of_year_sin"] = np.sin(2 * np.pi * hourly.index.dayofyear / 365.25)
hourly["day_of_year_cos"] = np.cos(2 * np.pi * hourly.index.dayofyear / 365.25)

if TARGET not in hourly.columns:
    raise ValueError(f"Target {TARGET!r} is missing after preprocessing.")

# Historical target temperature is intentionally included as an input feature.
FEATURE_COLUMNS = list(hourly.columns)
print("Hourly shape:", hourly.shape)
print("Feature count:", len(FEATURE_COLUMNS))
print("Features:", FEATURE_COLUMNS)


In [ ]:
train_rows = hourly.index < TRAIN_END
val_rows = (hourly.index >= TRAIN_END) & (hourly.index < TEST_START)
test_rows = (hourly.index >= TEST_START) & (hourly.index < TEST_END)

if train_rows.sum() == 0 or val_rows.sum() == 0 or test_rows.sum() == 0:
    raise ValueError(
        "The dataset does not contain the required 2009–2016 range. "
        f"Train={train_rows.sum()}, validation={val_rows.sum()}, test={test_rows.sum()}."
    )

feature_scaler = StandardScaler()
target_scaler = StandardScaler()

feature_scaler.fit(hourly.loc[train_rows, FEATURE_COLUMNS].dropna())
target_scaler.fit(hourly.loc[train_rows, [TARGET]].dropna())

scaled_features = feature_scaler.transform(hourly[FEATURE_COLUMNS])
scaled_target = target_scaler.transform(hourly[[TARGET]])[:, 0]
target_celsius = hourly[TARGET].to_numpy(dtype=np.float32)

print("Train rows:", int(train_rows.sum()))
print("Validation rows:", int(val_rows.sum()))
print("Test rows:", int(test_rows.sum()))


In [ ]:
def make_windows(index, features_scaled, target_scaled, target_celsius, history_hours, horizons):
    max_horizon = int(np.max(horizons))
    X_windows = []
    y_windows = []
    as_of_times = []
    target_times = []
    last_temperature_c = []

    for end_idx in range(history_hours - 1, len(index) - max_horizon):
        start_idx = end_idx - history_hours + 1
        stop_idx = end_idx + max_horizon + 1
        time_slice = index[start_idx:stop_idx]
        expected = pd.date_range(
            start=index[start_idx],
            periods=history_hours + max_horizon,
            freq="1h",
        )

        if not np.array_equal(time_slice.asi8, expected.asi8):
            continue

        window = features_scaled[start_idx:end_idx + 1]
        label_indices = end_idx + horizons
        labels = target_scaled[label_indices]

        if not np.isfinite(window).all() or not np.isfinite(labels).all():
            continue

        X_windows.append(window.astype(np.float32))
        y_windows.append(labels.astype(np.float32))
        as_of_times.append(index[end_idx])
        target_times.append(index[label_indices])
        last_temperature_c.append(float(target_celsius[end_idx]))

    return {
        "X": np.asarray(X_windows, dtype=np.float32),
        "y": np.asarray(y_windows, dtype=np.float32),
        "as_of_times": np.asarray(as_of_times, dtype="datetime64[ns]"),
        "target_times": np.asarray(target_times, dtype="datetime64[ns]"),
        "last_temperature_c": np.asarray(last_temperature_c, dtype=np.float32),
    }


all_windows = make_windows(
    hourly.index,
    scaled_features,
    scaled_target,
    target_celsius,
    HISTORY_HOURS,
    HORIZONS,
)

target_start = all_windows["target_times"][:, 0]
target_end = all_windows["target_times"][:, -1]

# Every label in a split must remain inside that split; this prevents boundary leakage.
split_masks = {
    "train": target_end < TRAIN_END.to_datetime64(),
    "validation": (
        (target_start >= TRAIN_END.to_datetime64())
        & (target_end < TEST_START.to_datetime64())
    ),
    "test": (
        (target_start >= TEST_START.to_datetime64())
        & (target_end < TEST_END.to_datetime64())
    ),
}


def select_windows(mask):
    return {
        key: value[mask] if isinstance(value, np.ndarray) else value
        for key, value in all_windows.items()
    }


train = select_windows(split_masks["train"])
validation = select_windows(split_masks["validation"])
test = select_windows(split_masks["test"])

print("Window shapes:")
for name, part in [("train", train), ("validation", validation), ("test", test)]:
    print(f"  {name}: X={part['X'].shape}, y={part['y'].shape}")
    if len(part["X"]):
        print(
            "    target range:",
            pd.Timestamp(part["target_times"][0, 0]),
            "to",
            pd.Timestamp(part["target_times"][-1, -1]),
        )

assert train["X"].shape[1:] == (HISTORY_HOURS, len(FEATURE_COLUMNS))
assert train["y"].shape[1] == len(HORIZONS)
assert len(test["X"]) > 0

for partition_name, partition in [("train", train), ("validation", validation), ("test", test)]:
    for horizon_index, horizon in enumerate(HORIZONS):
        expected_times = partition["as_of_times"] + np.timedelta64(int(horizon), "h")
        np.testing.assert_array_equal(
            partition["target_times"][:, horizon_index],
            expected_times,
            err_msg=f"{partition_name} timestamp alignment failed for +{horizon}h",
        )


In [ ]:
def as_dataset(part, shuffle):
    dataset = tf.data.Dataset.from_tensor_slices((part["X"], part["y"]))
    if shuffle:
        dataset = dataset.shuffle(
            buffer_size=min(len(part["X"]), 8192),
            seed=SEED,
            reshuffle_each_iteration=True,
        )
    return dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


train_dataset = as_dataset(train, shuffle=True)
validation_dataset = as_dataset(validation, shuffle=False)

inputs = tf.keras.Input(
    shape=(HISTORY_HOURS, len(FEATURE_COLUMNS)),
    name="weather_history",
)
x = tf.keras.layers.LSTM(
    64,
    return_sequences=True,
    dropout=0.20,
    name="lstm_64",
)(inputs)
x = tf.keras.layers.LayerNormalization(name="layer_norm")(x)
x = tf.keras.layers.LSTM(
    32,
    dropout=0.20,
    name="lstm_32",
)(x)
x = tf.keras.layers.Dense(
    16,
    activation="relu",
    kernel_regularizer=tf.keras.regularizers.l2(1e-4),
    name="regularized_dense",
)(x)
x = tf.keras.layers.Dropout(0.10, name="dense_dropout")(x)
outputs = tf.keras.layers.Dense(
    len(HORIZONS),
    name="temperature_forecast",
)(x)

model = tf.keras.Model(inputs, outputs, name="lstm_v2_multihorizon")
model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-3,
        clipnorm=1.0,
    ),
    loss=tf.keras.losses.Huber(),
    metrics=[tf.keras.metrics.MeanAbsoluteError(name="mae")],
)
model.summary()


In [ ]:
best_model_path = OUTPUT_DIR / "lstm_v2_best.keras"

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=8,
        restore_best_weights=True,
        mode="min",
        verbose=1,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        mode="min",
        verbose=1,
    ),
    tf.keras.callbacks.ModelCheckpoint(
        best_model_path,
        monitor="val_loss",
        save_best_only=True,
        mode="min",
        verbose=1,
    ),
]

history = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=MAX_EPOCHS,
    callbacks=callbacks,
    verbose=1,
)

print("Epochs completed:", len(history.history["loss"]))
print("Best validation loss:", float(np.min(history.history["val_loss"])))


In [ ]:
def inverse_target(values_scaled):
    values_scaled = np.asarray(values_scaled)
    original_shape = values_scaled.shape
    restored = target_scaler.inverse_transform(values_scaled.reshape(-1, 1))
    return restored.reshape(original_shape)


validation_true_c = inverse_target(validation["y"])
validation_pred_c = inverse_target(
    model.predict(validation["X"], batch_size=BATCH_SIZE, verbose=0)
)
test_true_c = inverse_target(test["y"])
test_pred_c = inverse_target(
    model.predict(test["X"], batch_size=BATCH_SIZE, verbose=0)
)

persistence_pred_c = np.repeat(
    test["last_temperature_c"][:, None],
    len(HORIZONS),
    axis=1,
)

metric_rows = []
for horizon_index, horizon in enumerate(HORIZONS):
    actual = test_true_c[:, horizon_index]
    predicted = test_pred_c[:, horizon_index]
    baseline = persistence_pred_c[:, horizon_index]

    model_mae = float(mean_absolute_error(actual, predicted))
    model_rmse = float(np.sqrt(mean_squared_error(actual, predicted)))
    baseline_mae = float(mean_absolute_error(actual, baseline))
    baseline_rmse = float(np.sqrt(mean_squared_error(actual, baseline)))

    metric_rows.append(
        {
            "model_version": "LSTM_v2",
            "horizon_hours": int(horizon),
            "MAE": model_mae,
            "RMSE": model_rmse,
            "persistence_MAE": baseline_mae,
            "persistence_RMSE": baseline_rmse,
            "v1_reference_MAE": V1_MAE_6H if horizon == 6 else np.nan,
            "v1_reference_RMSE": V1_RMSE_6H if horizon == 6 else np.nan,
            "parameters": int(model.count_params()),
            "epochs": len(history.history["loss"]),
        }
    )

metrics_df = pd.DataFrame(metric_rows)
display(metrics_df)

six_hour = metrics_df.loc[metrics_df["horizon_hours"] == 6].iloc[0]
beats_v1 = (
    six_hour["MAE"] < V1_MAE_6H
    and six_hour["RMSE"] < V1_RMSE_6H
)
beats_persistence = (
    six_hour["MAE"] < six_hour["persistence_MAE"]
    and six_hour["RMSE"] < six_hour["persistence_RMSE"]
)

if beats_v1 and beats_persistence:
    print("The +6h v2 result beats both the v1 reference and persistence baseline.")
else:
    print(
        "Do not claim v2 improvement yet: the +6h result does not beat both "
        "the v1 reference and the persistence baseline."
    )


In [ ]:
# Empirical residual calibration uses validation residuals only.
validation_residuals_c = validation_true_c - validation_pred_c

prediction_rows = []
threshold_rows = []

for horizon_index, horizon in enumerate(HORIZONS):
    residuals = validation_residuals_c[:, horizon_index]
    residual_q10, residual_q90 = np.quantile(residuals, [0.10, 0.90])

    for row_index in range(len(test_pred_c)):
        prediction = float(test_pred_c[row_index, horizon_index])
        actual = float(test_true_c[row_index, horizon_index])
        as_of = pd.Timestamp(test["as_of_times"][row_index])
        target_time = pd.Timestamp(test["target_times"][row_index, horizon_index])

        prediction_rows.append(
            {
                "as_of_timestamp": as_of,
                "target_timestamp": target_time,
                "horizon_hours": int(horizon),
                "actual_temperature_c": actual,
                "predicted_temperature_c": prediction,
                "lower_10_c": prediction + float(residual_q10),
                "upper_90_c": prediction + float(residual_q90),
            }
        )

        center_strike = int(np.rint(prediction))
        strikes = center_strike + np.arange(-2, 3)

        for strike in strikes:
            # Add-one smoothing keeps probabilities strictly inside (0, 1).
            successes = int(np.sum(prediction + residuals >= strike))
            probability_yes = (successes + 1) / (len(residuals) + 2)
            observed_yes = bool(actual >= strike)

            threshold_rows.append(
                {
                    "as_of_timestamp": as_of,
                    "target_timestamp": target_time,
                    "horizon_hours": int(horizon),
                    "strike_c": int(strike),
                    "actual_temperature_c": actual,
                    "predicted_temperature_c": prediction,
                    "probability_yes": float(probability_yes),
                    "observed_yes": observed_yes,
                    "brier_component": float(
                        (probability_yes - float(observed_yes)) ** 2
                    ),
                }
            )

predictions_df = pd.DataFrame(prediction_rows)
threshold_df = pd.DataFrame(threshold_rows)

assert len(predictions_df) == len(test["X"]) * len(HORIZONS)
assert threshold_df["probability_yes"].between(0.0, 1.0).all()

brier_df = (
    threshold_df.groupby("horizon_hours", as_index=False)["brier_component"]
    .mean()
    .rename(columns={"brier_component": "brier_score"})
)
metrics_df = metrics_df.merge(brier_df, on="horizon_hours", how="left")

calibration_bins = np.linspace(0.0, 1.0, 11)
threshold_df["probability_bin"] = pd.cut(
    threshold_df["probability_yes"],
    bins=calibration_bins,
    include_lowest=True,
)
calibration_summary_df = (
    threshold_df.groupby("probability_bin", observed=True)
    .agg(
        mean_probability=("probability_yes", "mean"),
        observed_rate=("observed_yes", "mean"),
        count=("observed_yes", "size"),
    )
    .reset_index()
)

display(metrics_df)
display(calibration_summary_df)


In [ ]:
# Keep the required deployment artifacts in /kaggle/working/.
model.save(OUTPUT_DIR / "lstm_v2_multihorizon.keras")
model.save(OUTPUT_DIR / "lstm_v2_multihorizon.h5")

joblib.dump(feature_scaler, OUTPUT_DIR / "feature_scaler_v2.pkl")
joblib.dump(target_scaler, OUTPUT_DIR / "target_scaler_v2.pkl")

predictions_df.to_csv(OUTPUT_DIR / "predictions_v2.csv", index=False)
threshold_df.drop(columns=["brier_component", "probability_bin"]).to_csv(
    OUTPUT_DIR / "threshold_predictions_v2.csv",
    index=False,
)
metrics_df.to_csv(OUTPUT_DIR / "experiment_results_v2.csv", index=False)
calibration_summary_df.to_csv(
    OUTPUT_DIR / "calibration_summary_v2.csv",
    index=False,
)

metadata = {
    "model_name": "lstm_v2_multihorizon",
    "model_version": "LSTM v2",
    "dataset": "Jena Climate 2009-2016",
    "target": TARGET,
    "unit": "degrees Celsius",
    "history_hours": HISTORY_HOURS,
    "horizons_hours": [int(value) for value in HORIZONS],
    "feature_columns": FEATURE_COLUMNS,
    "split": {
        "train_before": TRAIN_END.isoformat(),
        "validation_start": TRAIN_END.isoformat(),
        "test_start": TEST_START.isoformat(),
        "test_end": TEST_END.isoformat(),
    },
    "calibration_method": "empirical validation residuals with add-one smoothing",
    "synthetic_contracts": True,
    "contract_strikes": "round(prediction) plus [-2, -1, 0, 1, 2] degrees Celsius",
    "parameters": int(model.count_params()),
    "epochs_completed": len(history.history["loss"]),
    "v1_reference_6h": {
        "MAE": V1_MAE_6H,
        "RMSE": V1_RMSE_6H,
    },
    "training_timestamp_utc": pd.Timestamp.utcnow().isoformat(),
}

with open(OUTPUT_DIR / "model_metadata_v2.json", "w", encoding="utf-8") as metadata_file:
    json.dump(metadata, metadata_file, indent=2)

artifact_names = [
    "lstm_v2_multihorizon.keras",
    "lstm_v2_multihorizon.h5",
    "feature_scaler_v2.pkl",
    "target_scaler_v2.pkl",
    "predictions_v2.csv",
    "threshold_predictions_v2.csv",
    "experiment_results_v2.csv",
    "calibration_summary_v2.csv",
    "model_metadata_v2.json",
]

print("Exported artifacts:")
for name in artifact_names:
    path = OUTPUT_DIR / name
    print(f"  {name}: {path.stat().st_size:,} bytes")


## Interpretation and handoff

The exported threshold probabilities are synthetic research outputs. They are not official Kalshi probabilities because this notebook uses Jena Climate observations, not a Kalshi settlement feed or market prices.

For the future Streamlit v2 app:

- Use predictions_v2.csv for the multi-horizon forecast chart.
- Use threshold_predictions_v2.csv for the synthetic threshold ladder.
- Use experiment_results_v2.csv for metrics and persistence comparison.
- Use model_metadata_v2.json for the model-card and lineage panels.
- Keep the v1 app and v1 branch unchanged until the v2 artifacts have been reviewed.
